In [0]:
# TRUSTFLOW — NYC 311 SILVER PIPELINE
# Initialize Spark settings and historical table locations.
# Shared configuration for historical and incremental processing.
# Imports, configuration, single canonical table destinations
import json
from pathlib import Path
from datetime import date
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark.conf.set("spark.sql.session.timeZone", "America/New_York")
BRONZE = Path("/Volumes/workspace/default/trustflow_bronze/nyc_311")
SILVER_TABLE = "workspace.trustflow_silver.nyc_311_requests_v2"
QUARANTINE_TABLE = "workspace.trustflow_quality.nyc_311_quarantine_v2"
AUDIT_TABLE = "workspace.trustflow_quality.nyc_311_batch_audit_v2"

print("Bronze:", BRONZE)
print("Silver:", SILVER_TABLE)
print("Quarantine:", QUARANTINE_TABLE)
print("Audit:", AUDIT_TABLE)

Bronze: /Volumes/workspace/default/trustflow_bronze/nyc_311
Silver: workspace.trustflow_silver.nyc_311_requests_v2
Quarantine: workspace.trustflow_quality.nyc_311_quarantine_v2
Audit: workspace.trustflow_quality.nyc_311_batch_audit_v2


In [0]:
# TRUSTFLOW — EXECUTION CONFIGURATION
# Read preview/apply parameters from Databricks widgets.
# Validate approved table locations before allowing writes.

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# -------------------------------------------
# 1. EXECUTION PARAMETERS
# -------------------------------------------

dbutils.widgets.dropdown(
    "execution_mode",
    "preview",
    ["preview", "apply"]
)

dbutils.widgets.text(
    "write_confirmation",
    ""
)

execution_mode = dbutils.widgets.get("execution_mode")
write_confirmation = dbutils.widgets.get("write_confirmation")

RUN_MERGE = execution_mode == "apply"

# -------------------------------------------
# 2. DATA LOCATIONS
# -------------------------------------------

BRONZE = Path(
    "/Volumes/workspace/default/trustflow_bronze/nyc_311"
)

SILVER_TABLE = (
    "workspace.trustflow_silver.nyc_311_requests_v2"
)

QUARANTINE_TABLE = (
    "workspace.trustflow_quality.nyc_311_quarantine_v2"
)

REFRESH_AUDIT_TABLE = (
    "workspace.trustflow_quality.nyc_311_refresh_audit_v2"
)

spark.conf.set(
    "spark.sql.session.timeZone",
    "America/New_York"
)

# -------------------------------------------
# 3. WRITE SAFETY CHECKS
# -------------------------------------------

expected_tables = {
    "SILVER_TABLE":
        "workspace.trustflow_silver.nyc_311_requests_v2",
    "QUARANTINE_TABLE":
        "workspace.trustflow_quality.nyc_311_quarantine_v2",
    "REFRESH_AUDIT_TABLE":
        "workspace.trustflow_quality.nyc_311_refresh_audit_v2"
}

if RUN_MERGE:
    if write_confirmation != "APPLY_NYC311_ORIGINAL":
        raise RuntimeError(
            "Write mode requires explicit production confirmation."
        )

    for variable, expected in expected_tables.items():
        if globals()[variable] != expected:
            raise RuntimeError(
                f"Unsafe table configuration: {variable}"
            )

    print("MODE: APPLY — Original tables")
else:
    print("MODE: PREVIEW — Read only")

print("Silver:", SILVER_TABLE)
print("Quarantine:", QUARANTINE_TABLE)
print("Audit:", REFRESH_AUDIT_TABLE)


MODE: PREVIEW — Read only
Silver: workspace.trustflow_silver.nyc_311_requests_v2
Quarantine: workspace.trustflow_quality.nyc_311_quarantine_v2
Audit: workspace.trustflow_quality.nyc_311_refresh_audit_v2


In [0]:
# TRUSTFLOW — DELTA TABLE SCHEMAS
# Define Silver, Quarantine, and historical Audit schemas.
# Verify compatibility with incremental tracking columns.

SILVER_FIELDS = [
    ("unique_key", "string"),
    ("created_date", "timestamp"),
    ("closed_date", "timestamp"),
    ("resolution_action_updated_date", "timestamp"),
    ("due_date", "timestamp"),
    ("status", "string"),
    ("complaint_type", "string"),
    ("descriptor", "string"),
    ("agency", "string"),
    ("agency_name", "string"),
    ("borough", "string"),
    ("incident_zip", "string"),
    ("city", "string"),
    ("location_type", "string"),
    ("resolution_description", "string"),
    ("latitude", "double"),
    ("longitude", "double"),
    ("ingestion_batch_id", "string"),
    ("loaded_at", "timestamp"),
]

QUARANTINE_FIELDS = [
    ("batch_id", "string"),
    ("unique_key", "string"),
    ("failure_reason", "string"),
    ("record_json", "string"),
    ("record_sha256", "string"),
    ("quarantined_at", "timestamp"),
]

AUDIT_FIELDS = [
    ("batch_id", "string"),
    ("process_date", "string"),
    ("bronze_count", "long"),
    ("valid_count", "long"),
    ("quarantine_count", "long"),
    ("status", "string"),
    ("processed_at", "timestamp"),
]

def as_schema(fields):
    return T.StructType([
        T.StructField(name, {"string":T.StringType(),"timestamp":T.TimestampType(),"double":T.DoubleType(),"long":T.LongType()}[dtype], True)
        for name, dtype in fields
    ])

required_tables = [
    SILVER_TABLE,
    QUARANTINE_TABLE,
    AUDIT_TABLE,
]

for table_name in required_tables:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table is missing: {table_name}. "
            "Run one-time setup before processing."
        )

print("All required historical Delta tables exist.")


# Validate existing Delta table schemas.
# Silver may include the additional incremental tracking column.

expected_schemas = {
    SILVER_TABLE: SILVER_FIELDS,
    QUARANTINE_TABLE: QUARANTINE_FIELDS,
    AUDIT_TABLE: AUDIT_FIELDS,
}

for table_name, required_fields in expected_schemas.items():

    actual = {
        field.name: field.dataType.simpleString()
        for field in spark.table(table_name).schema.fields
    }

    expected = {
        field.name: field.dataType.simpleString()
        for field in as_schema(required_fields).fields
    }

    # Check that every required column exists with the correct type.
    for name, dtype in expected.items():
        if name not in actual:
            raise RuntimeError(
                f"Missing column {name} in {table_name}"
            )

        if actual[name] != dtype:
            raise RuntimeError(
                f"Incorrect type for {name} in {table_name}: "
                f"expected {dtype}, found {actual[name]}"
            )

    # Allow only our known additional Silver tracking column.
    allowed_extra = (
        {"last_observed_at": "timestamp"}
        if table_name == SILVER_TABLE
        else {}
    )

    extras = set(actual) - set(expected)

    if extras != (extras & set(allowed_extra)):
        raise RuntimeError(
            f"Unexpected extra columns in {table_name}: "
            f"{sorted(extras - set(allowed_extra))}"
        )

    for name in extras:
        if actual[name] != allowed_extra[name]:
            raise RuntimeError(
                f"Incorrect type for {name} in {table_name}"
            )

print("All 3 Delta table schemas are compatible.")


All required historical Delta tables exist.
All 3 Delta table schemas are compatible.


In [0]:
# TRUSTFLOW — HISTORICAL BATCH DISCOVERY
# Find completed Bronze batches using metadata.
# Skip historical batches already marked successful.

def completed_bronze_batches():
    metadata_dir = BRONZE / "metadata" / "historical"
    if not metadata_dir.is_dir():
        raise FileNotFoundError(f"Metadata directory missing: {metadata_dir}")

    batches = []
    for manifest_path in sorted(metadata_dir.glob("*.json")):
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("status") != "SUCCESS":
            continue
        day = manifest_path.stem
        date.fromisoformat(day)
        if manifest.get("source") != "nyc_311" or manifest.get("date") != day:
            raise ValueError(f"Invalid manifest: {manifest_path}")
        rows = manifest.get("records_downloaded")
        pages = manifest.get("pages_downloaded")
        if not isinstance(rows, int) or not isinstance(pages, int) or rows <= 0 or pages <= 0:
            raise ValueError(f"Unexpected empty or invalid SUCCESS batch: {day}")
        batch_dir = BRONZE / "historical" / day
        files = sorted(batch_dir.glob("page_*.json"))
        if len(files) != pages:
            raise ValueError(f"Page-count mismatch for {day}: expected={pages}, actual={len(files)}")
        batches.append((day, rows, [str(p) for p in files]))
    return batches

processed = {
    row.batch_id
    for row in spark.table(AUDIT_TABLE)
        .filter(F.col("status") == "SUCCESS")
        .select("batch_id").collect()
}

pending_batches = [
    batch for batch in completed_bronze_batches()
    if f"nyc_311_historical_{batch[0]}" not in processed
]
print("Pending completed batches:", len(pending_batches))
print("Dates:", [batch[0] for batch in pending_batches])

Pending completed batches: 0
Dates: []


In [0]:
# TRUSTFLOW — HISTORICAL SILVER PROCESSING
# Validate raw records and quarantine invalid data.
# Insert new valid records using Delta MERGE.
# Record successful batches in the historical Audit.

def source_col(df, name, dtype):
    if name not in df.columns:
        return F.lit(None).cast(dtype)
    value = F.col(f"`{name}`")
    if dtype in ("timestamp", "double"):
        return F.expr(f"try_cast(`{name}` AS {dtype})")
    return F.trim(value.cast("string"))


def process_batch(day, expected_count, paths):
    batch_id = f"nyc_311_historical_{day}"

    # Re-check current audit state; lets repeated notebook runs skip completed batches.
    if spark.table(AUDIT_TABLE).filter(
        (F.col("batch_id") == batch_id) & (F.col("status") == "SUCCESS")
    ).limit(1).count():
        print("SKIP already processed:", batch_id)
        return

    raw = spark.read.option("multiLine", "true").json(paths)
    bronze_count = raw.count()
    if bronze_count != expected_count:
        raise RuntimeError(f"{day}: Bronze mismatch: expected {expected_count}, read {bronze_count}")

    # Source JSON is kept intact in the Bronze Volume; this is a parsed JSON representation.
    raw_json = F.to_json(F.struct(*[F.col(f"`{c}`") for c in raw.columns]))
    staged = raw.withColumn("_record_json", raw_json)

    for name, dtype in SILVER_FIELDS:
        if name in ("ingestion_batch_id", "loaded_at"):
            continue
        staged = staged.withColumn(f"_silver_{name}", source_col(raw, name, dtype))

    key = F.col("_silver_unique_key")
    created = F.col("_silver_created_date")
    closed = F.col("_silver_closed_date")
    duplicate_keys = (
        staged.filter(key.isNotNull() & (key != ""))
        .groupBy("_silver_unique_key")
        .count().filter(F.col("count") > 1)
        .select("_silver_unique_key")
        .withColumn("_duplicate", F.lit(True))
    )
    staged = staged.join(duplicate_keys, "_silver_unique_key", "left")

    reasons = F.concat_ws(",",
        F.when(key.isNull() | (key == ""), F.lit("MISSING_REQUEST_ID")),
        F.when(created.isNull(), F.lit("MISSING_OR_INVALID_CREATED_DATE")),
        F.when(created.isNotNull() & closed.isNotNull() & (closed < created), F.lit("CLOSED_BEFORE_CREATED")),
        F.when(F.col("_duplicate").isNotNull(), F.lit("DUPLICATE_ID_IN_BATCH")),
    )
    staged = staged.withColumn("_failure_reason", reasons)
    rejected = staged.filter(F.col("_failure_reason") != "")
    accepted = staged.filter(F.col("_failure_reason") == "")

    # Build stable table schemas regardless of optional fields missing in source JSON.
    silver = accepted.select(*[
        (F.lit(batch_id).alias(name) if name == "ingestion_batch_id" else
         F.current_timestamp().alias(name) if name == "loaded_at" else
         F.col(f"_silver_{name}").alias(name))
        for name, _ in SILVER_FIELDS
    ])
    quarantine = rejected.select(
        F.lit(batch_id).alias("batch_id"),
        F.col("_silver_unique_key").alias("unique_key"),
        F.col("_failure_reason").alias("failure_reason"),
        F.col("_record_json").alias("record_json"),
        F.sha2(F.col("_record_json"), 256).alias("record_sha256"),
        F.current_timestamp().alias("quarantined_at"),
    )

    valid_count = silver.count()
    quarantine_count = quarantine.count()
    if bronze_count != valid_count + quarantine_count:
        raise RuntimeError(f"{day}: Reconciliation failed: {bronze_count} != {valid_count} + {quarantine_count}")

    # Atomic overwrite for this one quarantine partition; safe on retries.
    (quarantine.write.format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"batch_id = '{batch_id}'")
        .saveAsTable(QUARANTINE_TABLE))

    # Historical insert-only MERGE.
    # Insert only the historical fields.
    # Leave last_observed_at NULL for historical records.
    insert_values = {
        name: f"s.`{name}`"
        for name, _ in SILVER_FIELDS
    }

    (
        DeltaTable.forName(spark, SILVER_TABLE)
        .alias("t")
        .merge(
            silver.alias("s"),
            "t.unique_key = s.unique_key"
    )
    .whenNotMatchedInsert(values=insert_values)
    .execute()
    )


    # Audit recorded LAST. If a failure occurs earlier, retry is idempotent.
    audit = spark.createDataFrame(
        [(batch_id, day, bronze_count, valid_count, quarantine_count, "SUCCESS")],
        "batch_id string, process_date string, bronze_count long, valid_count long, quarantine_count long, status string"
    ).withColumn("processed_at", F.current_timestamp())
    DeltaTable.forName(spark, AUDIT_TABLE).alias("t").merge(
        audit.alias("s"), "t.batch_id = s.batch_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

    print(f"SUCCESS {day}: Bronze={bronze_count}, Valid={valid_count}, Quarantine={quarantine_count}")

In [0]:

# TRUSTFLOW — HISTORICAL BATCH EXECUTION
# Process pending batches in chronological order.
# Skip all historical writes in preview mode.

if RUN_MERGE:
    for day, expected_count, paths in pending_batches:
        process_batch(day, expected_count, paths)

    print("Historical processing completed.")
else:
    print(
        "PREVIEW MODE: Historical processing skipped. "
        f"Pending batches: {len(pending_batches)}"
    )


PREVIEW MODE: Historical processing skipped. Pending batches: 0


In [0]:

# TRUSTFLOW — HISTORICAL DATA QUALITY
# Verify batch reconciliation and historical Quarantine.
# Check Silver IDs and required fields without modifying data.

from pyspark.sql import functions as F

silver = spark.table(SILVER_TABLE)
quarantine = spark.table(QUARANTINE_TABLE)
audit = spark.table(AUDIT_TABLE)

# 1. Check historical batch reconciliation.
successful = audit.filter(F.col("status") == "SUCCESS")

failed_reconciliation = successful.filter(
    F.col("bronze_count") != (
        F.col("valid_count") + F.col("quarantine_count")
    )
).count()

# 2. Check duplicate Silver request IDs.
duplicate_ids = (
    silver.groupBy("unique_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# 3. Check required Silver fields.
invalid_keys = silver.filter(
    F.col("unique_key").isNull()
    | (F.trim(F.col("unique_key")) == "")
    | F.col("created_date").isNull()
).count()

# 4. Separate historical and incremental Quarantine.
historical_quarantine = quarantine.filter(
    F.col("batch_id").startswith("nyc_311_historical_")
)

refresh_quarantine = quarantine.filter(
    F.col("batch_id").startswith("nyc_311_refresh_")
)

# 5. Check historical Quarantine against each successful batch.
historical_quarantine_counts = (
    historical_quarantine
    .groupBy("batch_id")
    .agg(F.count("*").alias("actual_quarantine"))
)

batch_check = (
    successful
    .join(historical_quarantine_counts, "batch_id", "left")
    .withColumn(
        "actual_quarantine",
        F.coalesce(F.col("actual_quarantine"), F.lit(0))
    )
)

quarantine_mismatches = batch_check.filter(
    F.col("quarantine_count") != F.col("actual_quarantine")
).count()

# Also detect historical quarantine batches lacking
# a successful audit record.
untracked_quarantine_batches = (
    historical_quarantine_counts
    .join(
        successful.select("batch_id"),
        "batch_id",
        "left_anti"
    )
    .count()
)

# 6. Calculate summary numbers.
totals = successful.agg(
    F.coalesce(F.sum("bronze_count"), F.lit(0)).alias("bronze"),
    F.coalesce(F.sum("valid_count"), F.lit(0)).alias("valid"),
    F.coalesce(F.sum("quarantine_count"), F.lit(0)).alias("invalid")
).first()

silver_count = silver.count()
historical_quarantine_count = historical_quarantine.count()
refresh_quarantine_count = refresh_quarantine.count()

print("Historical batches:", successful.count())
print("Historical Bronze records:", totals["bronze"])
print("Historical valid records:", totals["valid"])
print("Historical Quarantine records:", historical_quarantine_count)
print("Refresh Quarantine records:", refresh_quarantine_count)
print("Current Silver records:", silver_count)

print("Failed historical reconciliations:", failed_reconciliation)
print("Quarantine batch mismatches:", quarantine_mismatches)
print("Untracked historical quarantine batches:",
      untracked_quarantine_batches)
print("Duplicate Silver IDs:", duplicate_ids)
print("Invalid Silver keys:", invalid_keys)

# 7. Final quality checks.
assert failed_reconciliation == 0
assert quarantine_mismatches == 0
assert untracked_quarantine_batches == 0
assert duplicate_ids == 0
assert invalid_keys == 0
assert totals["bronze"] == totals["valid"] + totals["invalid"]
assert historical_quarantine_count == totals["invalid"]

print("HISTORICAL QUALITY GATE PASSED")


Historical batches: 13
Historical Bronze records: 140605
Historical valid records: 140473
Historical Quarantine records: 132
Refresh Quarantine records: 132
Current Silver records: 140528
Failed historical reconciliations: 0
Quarantine batch mismatches: 0
Untracked historical quarantine batches: 0
Duplicate Silver IDs: 0
Invalid Silver keys: 0
HISTORICAL QUALITY GATE PASSED


In [0]:
# TRUSTFLOW — INCREMENTAL REFRESH DISCOVERY
# Discover completed API refresh snapshots automatically.
# Validate metadata, observation timestamps, and page counts.

metadata_folder = BRONZE / "metadata" / "refresh"
snapshots = []

for file in sorted(metadata_folder.glob("*.json")):
    meta = json.loads(file.read_text(encoding="utf-8"))

    if meta.get("status") != "SUCCESS":
        continue

    day = file.stem
    datetime.strptime(day, "%Y-%m-%d")

    if (
        meta.get("source") != "nyc_311"
        or meta.get("mode") != "refresh"
        or meta.get("date") != day
    ):
        raise RuntimeError(f"Invalid metadata: {file}")

    observed = meta.get("observed_at")
    if not observed:
        raise RuntimeError(f"Missing observed_at: {file}")

    observed_dt = datetime.fromisoformat(
        observed.replace("Z", "+00:00")
    )

    if observed_dt.tzinfo is None:
        raise RuntimeError(
            f"observed_at must include a timezone: {file}"
        )

    pages = sorted(
        (BRONZE / "refresh" / day).glob("page_*.json")
    )

    if len(pages) != int(meta["pages_downloaded"]):
        raise RuntimeError(f"Missing pages for {day}")

    snapshots.append((day, observed, meta, pages))

if not snapshots:
    raise RuntimeError("No completed refresh snapshots found")


In [0]:
# TRUSTFLOW — REFRESH AUDIT TRACKING
# Identify completed and interrupted refresh batches.
# Prevent duplicate processing and track observation times.

audit_exists = spark.catalog.tableExists(
    REFRESH_AUDIT_TABLE
)

# Verify the refresh Audit table exists before processing.
if not audit_exists:
    raise RuntimeError(
        f"Missing refresh Audit table: {REFRESH_AUDIT_TABLE}. "
        "Run one-time setup first."
    )

if audit_exists:
    required_audit_columns = {
        "batch_id", "snapshot_date", "observed_at", "manifest_sha256",
        "status", "source_count", "valid_count", "quarantine_count",
        "planned_insert", "planned_update", "planned_skip",
        "started_at", "completed_at"
    }
    missing_audit_columns = required_audit_columns - set(
        spark.table(REFRESH_AUDIT_TABLE).columns
    )
    if missing_audit_columns:
        raise RuntimeError(
            "Existing refresh audit has incompatible schema: "
            f"{sorted(missing_audit_columns)}. Do not overwrite it."
        )

completed = set()
newest_completed_at = None

if audit_exists:
    audit_rows = (
        spark.table(REFRESH_AUDIT_TABLE)
        .select("batch_id", "snapshot_date", "observed_at", "manifest_sha256", "status")
        .collect()
    )

    # Refuse duplicates or conflicting snapshots for the same batch ID.
    audit_by_batch = {}
    for r in audit_rows:
        if r.batch_id in audit_by_batch:
            raise RuntimeError(f"Duplicate audit batch ID: {r.batch_id}")
        audit_by_batch[r.batch_id] = r

    completed = {
        r.batch_id for r in audit_rows if r.status == "SUCCESS"
    }
    successful_times = [
        datetime.fromisoformat(r.observed_at.replace("Z", "+00:00"))
        for r in audit_rows if r.status == "SUCCESS"
    ]
    if successful_times:
        if any(t.tzinfo is None for t in successful_times):
            raise RuntimeError("Audit contains a timezone-less observation")
        newest_completed_at = max(successful_times)
else:
    audit_by_batch = {}



In [0]:
# TRUSTFLOW — INCREMENTAL SILVER MERGE
# Validate refreshed records and detect field-level changes.
# Insert new records, update changed records, quarantine invalid data.
# Verify MERGE results and record successful batch execution.

# SAFETY GUARD — Automated execution mode

if RUN_MERGE:
    if (
        execution_mode != "apply"
        or write_confirmation != "APPLY_NYC311_ORIGINAL"
    ):
        raise RuntimeError(
            "Production write authorization failed."
        )

    expected = {
        "SILVER_TABLE":
            "workspace.trustflow_silver.nyc_311_requests_v2",
        "QUARANTINE_TABLE":
            "workspace.trustflow_quality.nyc_311_quarantine_v2",
        "REFRESH_AUDIT_TABLE":
            "workspace.trustflow_quality.nyc_311_refresh_audit_v2"
    }

    for variable, table_name in expected.items():
        if globals()[variable] != table_name:
            raise RuntimeError(
                f"Unsafe write target: {variable}"
            )

    print("Original table write configuration verified.")
else:
    print("PREVIEW MODE — No table writes authorized.")


pending = [
    item for item in snapshots
    if f"nyc_311_refresh_{item[0]}" not in completed
]

pending.sort(key=lambda item: datetime.fromisoformat(
    item[1].replace("Z", "+00:00")
))

if not pending:
    print("No pending refresh snapshots. Nothing to do.")

if not RUN_MERGE:
    print("PREVIEW MODE — Silver will not be changed.")
    if len(pending) > 1:
        print("Previewing the first pending snapshot only.")
    pending = pending[:1]

for day, observed, meta, pages in pending:

    batch_id = f"nyc_311_refresh_{day}"

    observed_dt = datetime.fromisoformat(
        observed.replace("Z", "+00:00")
    )

    # An incomplete previous attempt is allowed, but it must refer
    # to the same immutable snapshot. Do not silently replace its input.
    # Manifest identity makes a resumed STARTED batch reject edited metadata.
    manifest_sha256 = hashlib.sha256(
        (metadata_folder / f"{day}.json").read_bytes()
    ).hexdigest()
    previous = audit_by_batch.get(batch_id)
    if previous is not None:
        if (previous.snapshot_date != day
                or previous.observed_at != observed
                or previous.manifest_sha256 != manifest_sha256):
            raise RuntimeError(
                f"Snapshot metadata changed after an audit began: {batch_id}"
            )
        if previous.status != "STARTED":
            raise RuntimeError(
                f"Unexpected audit state for {batch_id}: {previous.status}"
            )

    # Compare actual observation times, not only folder dates.
    if newest_completed_at and observed_dt <= newest_completed_at:
        raise RuntimeError(
            f"Snapshot {day} is not newer than the latest SUCCESS batch. "
            "Do not apply stale data."
        )

    print(f"\nProcessing snapshot: {day}")

    # Read and verify Bronze.
    raw = (
        spark.read
        .option("multiLine", "true")
        .json([str(p) for p in pages])
    )

    source_count = raw.count()

    if (source_count != int(meta["records_downloaded"])
            or source_count != int(meta["expected_records"])):
        raise RuntimeError(
            f"Bronze count mismatch for {day}"
        )

    # Validate request IDs before MERGE.
    key = F.trim(F.col("unique_key").cast("string"))

    invalid_keys = raw.filter(
        key.isNull() | (key == "")
    ).count()

    duplicate_keys = (
        raw.select(key.alias("unique_key"))
        .groupBy("unique_key")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    if invalid_keys or duplicate_keys:
        raise RuntimeError(
            f"Invalid or duplicate request IDs: "
            f"missing={invalid_keys}, "
            f"duplicate_groups={duplicate_keys}"
        )

    # The first refresh starts from a historical baseline with unknown
    # per-record observation time (NULL). Do NOT fabricate timestamps.
    # Later changed/inserted rows get a true refresh observation time.
    # The global successful-snapshot watermark above prevents replaying
    # an older batch after a newer batch succeeds.
    target = spark.table(SILVER_TABLE)

    if RUN_MERGE:
        dup_target = (
            target.groupBy("unique_key")
            .count().filter(F.col("count") > 1)
            .limit(1).count()
        )
        if dup_target:
            raise RuntimeError("Duplicate unique_key values in Silver")

    excluded = {
        "ingestion_batch_id",
        "loaded_at",
        "last_observed_at"
    }

    fields = [
        f for f in target.schema.fields
        if f.name not in excluded
    ]

    names = [f.name for f in fields]

    # Detect source columns missing from the refresh.
    missing_source_columns = [
        name for name in names
        if name not in raw.columns
    ]

    if missing_source_columns:
        raise RuntimeError(
            "Refresh is missing Silver columns. "
            "Review before updating: "
            f"{missing_source_columns}"
        )

    # Normalize exactly as in historical Silver.
    expressions = []

    for field in fields:
        name = field.name
        dtype = field.dataType.simpleString()

        if name not in raw.columns:
            value = F.lit(None).cast(field.dataType)
        elif dtype == "string":
            value = F.trim(
                F.col(f"`{name}`").cast("string")
            )
        else:
            value = F.expr(
                f"try_cast(`{name}` AS {dtype})"
            )

        expressions.append(value.alias(name))

    original_json = F.to_json(
        F.struct(
            *[F.col(f"`{c}`") for c in raw.columns]
        )
    )

    prepared = raw.select(
        *expressions,
        original_json.alias("_raw_json")
    )

    # Apply the same quality rules as historical Silver.
    prepared = prepared.withColumn(
        "_failure_reason",
        F.concat_ws(
            ",",
            F.when(
                F.col("unique_key").isNull()
                | (F.col("unique_key") == ""),
                "MISSING_REQUEST_ID"
            ),
            F.when(
                F.col("created_date").isNull(),
                "MISSING_OR_INVALID_CREATED_DATE"
            ),
            F.when(
                F.col("created_date").isNotNull()
                & F.col("closed_date").isNotNull()
                & (
                    F.col("closed_date")
                    < F.col("created_date")
                ),
                "CLOSED_BEFORE_CREATED"
            )
        )
    )

    valid = (
        prepared
        .filter(F.col("_failure_reason") == "")
        .select(*names)
    )

    invalid = prepared.filter(
        F.col("_failure_reason") != ""
    )

    valid_count = valid.count()
    invalid_count = invalid.count()

    if valid_count + invalid_count != source_count:
        raise RuntimeError("Validation reconciliation failed")

    # Calculate fingerprints for full-record comparison.
    def fingerprint(df):
        return F.sha2(
            F.to_json(
                F.struct(
                    *[F.col(f"`{n}`") for n in names]
                ),
                options={"ignoreNullFields": "false"}
            ),
            256
        )

    valid = valid.withColumn(
        "_source_hash", fingerprint(valid)
    )

    existing = (
        target
        .select(*names)
        .withColumn("_target_hash", fingerprint(target))
        .select(
            "unique_key",
            "_target_hash",
            F.lit(True).alias("_exists")
        )
    )

    plan = (
        valid
        .join(existing, on="unique_key", how="left")
        .withColumn(
            "_action",
            F.when(
                F.col("_exists").isNull(),
                "INSERT"
            )
            .when(
                ~F.col("_source_hash").eqNullSafe(
                    F.col("_target_hash")
                ),
                "UPDATE"
            )
            .otherwise("SKIP")
        )
    )

    counts = {
        r["_action"]: r["count"]
        for r in plan.groupBy("_action").count().collect()
    }

    inserts = counts.get("INSERT", 0)
    updates = counts.get("UPDATE", 0)
    skips = counts.get("SKIP", 0)

    if inserts + updates + skips + invalid_count != source_count:
        raise RuntimeError("Action reconciliation failed")

    print("INSERT:", inserts)
    print("UPDATE:", updates)
    print("SKIP:", skips)
    print("QUARANTINE:", invalid_count)
    print("TOTAL:", source_count)

    if not RUN_MERGE:
        print("Preview complete. No tables modified.")
        continue

    # --------------------------------------------------
    # 5. RECORD AUDIT BEFORE WRITING
    # --------------------------------------------------

    # Prevent parallel refresh writers: orchestrate this notebook with
    # max concurrent runs = 1. Delta cannot atomically commit the
    # Silver, quarantine, and audit tables as a single transaction.
    audit_start = spark.createDataFrame(
        [(
            batch_id, day, observed, manifest_sha256,
            source_count, valid_count, invalid_count,
            inserts, updates, skips, "STARTED"
        )],
        """
        batch_id STRING,
        snapshot_date STRING,
        observed_at STRING,
        manifest_sha256 STRING,
        source_count LONG,
        valid_count LONG,
        quarantine_count LONG,
        planned_insert LONG,
        planned_update LONG,
        planned_skip LONG,
        status STRING
        """
    ).withColumn(
        "started_at", F.current_timestamp()
    ).withColumn(
        "completed_at", F.lit(None).cast("timestamp")
    )

    (
        DeltaTable.forName(spark, REFRESH_AUDIT_TABLE)
        .alias("t")
        .merge(
            audit_start.alias("s"),
            "t.batch_id = s.batch_id"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    # --------------------------------------------------
    # 6. SAVE INVALID RECORDS TO QUARANTINE
    # --------------------------------------------------

    quarantine_df = invalid.select(
        F.lit(batch_id).alias("batch_id"),
        F.col("unique_key"),
        F.col("_failure_reason").alias("failure_reason"),
        F.col("_raw_json").alias("record_json"),
        F.sha2(F.col("_raw_json"), 256)
            .alias("record_sha256"),
        F.current_timestamp().alias("quarantined_at")
    )

    (
        quarantine_df.write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"batch_id = '{batch_id}'"
        )
        .saveAsTable(QUARANTINE_TABLE)
    )

    # Verify Quarantine batch contents were written before touching Silver.
    stored_invalid = spark.table(QUARANTINE_TABLE).filter(
        F.col("batch_id") == batch_id
    )
    if stored_invalid.count() != invalid_count:
        raise RuntimeError("Quarantine count verification failed")
    # Distinct-key check protects against a malformed retry write.
    if stored_invalid.select("unique_key").distinct().count() != invalid_count:
        raise RuntimeError("Quarantine unique ID verification failed")

    # --------------------------------------------------
    # 7. SAFE DELTA MERGE
    # --------------------------------------------------

    if "last_observed_at" not in target.columns:
        spark.sql(
            f"ALTER TABLE {SILVER_TABLE} "
            "ADD COLUMNS (last_observed_at TIMESTAMP)"
        )

    # Databricks parses an offset-bearing ISO datetime as an instant.
    # Avoid assigning the current processing time to old source data.
    merge_source = valid.drop("_source_hash").withColumn(
        "_observed_at",
        F.to_timestamp(F.lit(observed))
    )
    if merge_source.filter(F.col("_observed_at").isNull()).limit(1).count():
        raise RuntimeError("Could not parse refresh observed_at timestamp")

    # Exact, null-safe comparison of all data fields.
    different = " OR ".join(
        f"NOT (t.`{n}` <=> s.`{n}`)"
        for n in names
    )

    changes = {
        n: f"s.`{n}`"
        for n in names
    }

    changes.update({
        "ingestion_batch_id": "s._batch_id",
        "loaded_at": "current_timestamp()",
        "last_observed_at": "s._observed_at"
    })

    merge_source = merge_source.withColumn(
        "_batch_id", F.lit(batch_id)
    )

    # A stale target with a *different* value is unsafe; if source and
    # target already match (e.g. retry after a partial run), it is fine.
    target_for_conflicts = spark.table(SILVER_TABLE)
    current_hashes = (
        target_for_conflicts
        .withColumn("_old_hash", fingerprint(target_for_conflicts))
        .select("unique_key", "last_observed_at", "_old_hash")
    )
    conflicts = (
        valid.select("unique_key", "_source_hash")
        .join(current_hashes, "unique_key", "inner")
        .filter(
            (F.col("last_observed_at") >= F.to_timestamp(F.lit(observed)))
            & (~F.col("_source_hash").eqNullSafe(F.col("_old_hash")))
        ).limit(1).count()
    )
    if conflicts:
        raise RuntimeError(
            "A newer or equal-time target record conflicts with this snapshot. "
            "Do not overwrite it."
        )

    before_count = spark.table(SILVER_TABLE).count()

    (
        DeltaTable.forName(spark, SILVER_TABLE)
        .alias("t")
        .merge(
            merge_source.alias("s"),
            "t.unique_key = s.unique_key"
        )
        .whenMatchedUpdate(
            condition=(
                "(t.last_observed_at IS NULL OR "
                "s._observed_at > t.last_observed_at) "
                f"AND ({different})"
            ),
            set=changes
        )
        .whenNotMatchedInsert(values=changes)
        .execute()
    )

    # --------------------------------------------------
    # 8. VERIFY THE RESULT
    # --------------------------------------------------

    after_count = spark.table(SILVER_TABLE).count()

    if after_count != before_count + inserts:
        raise RuntimeError(
            "Silver row count verification failed"
        )

    updated_target = spark.table(SILVER_TABLE)

    target_hashes = (
        updated_target
        .withColumn(
            "_actual_hash",
            fingerprint(updated_target)
        )
        .select("unique_key", "_actual_hash")
    )

    mismatches = (
        valid
        .select("unique_key", "_source_hash")
        .join(target_hashes, on="unique_key", how="left")
        .filter(
            ~F.col("_source_hash").eqNullSafe(
                F.col("_actual_hash")
            )
        )
        .count()
    )

    if mismatches:
        raise RuntimeError(
            f"Post-MERGE verification failed: "
            f"{mismatches} mismatched records"
        )

    # Ensure invalid versions weren't inserted by this refresh.
    # (Existing last-valid Silver versions are intentionally retained.)
    print("Verified valid source rows:", valid_count)
    print("Verified quarantine rows:", invalid_count)

    # Mark the batch successful only after all checks.
    spark.sql(f"""
        UPDATE {REFRESH_AUDIT_TABLE}
        SET status = 'SUCCESS',
            completed_at = current_timestamp()
        WHERE batch_id = '{batch_id}'
          AND status = 'STARTED'
    """)

    newest_completed_at = observed_dt

    print("MERGE SUCCESS:", batch_id)
    print("Silver records before:", before_count)
    print("Silver records after:", after_count)
    print("Verified mismatches:", mismatches)

print("\nRefresh processing finished.")


PREVIEW MODE — No table writes authorized.
No pending refresh snapshots. Nothing to do.
PREVIEW MODE — Silver will not be changed.

Refresh processing finished.


In [0]:

# TRUSTFLOW — FINAL INCREMENTAL QUALITY CHECK
# Verify Silver uniqueness, Audit reconciliation, and Quarantine.
# Use dynamic batch counts; no hardcoded dates or totals.
# READ ONLY.

from pyspark.sql import functions as F

SILVER_TABLE = "workspace.trustflow_silver.nyc_311_requests_v2"

QUARANTINE_TABLE = (
    "workspace.trustflow_quality.nyc_311_quarantine_v2"
)

REFRESH_AUDIT_TABLE = (
    "workspace.trustflow_quality.nyc_311_refresh_audit_v2"
)

silver = spark.table(SILVER_TABLE)
quarantine = spark.table(QUARANTINE_TABLE)
audit = spark.table(REFRESH_AUDIT_TABLE)

# 1. Validate the current Silver table.
silver_count = silver.count()

duplicate_ids = (
    silver.groupBy("unique_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_silver = silver.filter(
    F.col("unique_key").isNull()
    | (F.trim(F.col("unique_key")) == "")
    | F.col("created_date").isNull()
).count()

print("Current Silver records:", silver_count)
print("Duplicate Silver IDs:", duplicate_ids)
print("Invalid Silver records:", invalid_silver)

assert duplicate_ids == 0
assert invalid_silver == 0

# 2. Read completed refresh batches automatically.
successful = audit.filter(F.col("status") == "SUCCESS")

successful_count = successful.count()

print("Successful refresh batches:", successful_count)

if successful_count == 0:
    print("No completed refresh batches to verify.")

else:
    # 3. Check recorded reconciliation for each completed batch.
    bad_reconciliation = successful.filter(
    (
        F.col("source_count") != (
            F.col("valid_count") +
            F.col("quarantine_count")
        )
    )
    |
    (
        F.col("valid_count") != (
            F.col("planned_insert") +
            F.col("planned_update") +
            F.col("planned_skip")
        )
    )
    ).count()
    
    # 4. Compare stored Quarantine counts with Audit.
    quarantine_counts = (
        quarantine
        .filter(
            F.col("batch_id").startswith("nyc_311_refresh_")
        )
        .groupBy("batch_id")
        .agg(F.count("*").alias("actual_quarantine"))
    )

    batch_checks = (
        successful
        .join(quarantine_counts, "batch_id", "left")
        .withColumn(
            "actual_quarantine",
            F.coalesce(
                F.col("actual_quarantine"),
                F.lit(0)
            )
        )
    )

    quarantine_mismatches = batch_checks.filter(
        F.col("quarantine_count")
        != F.col("actual_quarantine")
    ).count()

    # 5. Show the latest successful refresh.
    latest = (
        successful
        .orderBy(F.col("completed_at").desc())
        .limit(1)
        .first()
    )

    print("\nLatest successful refresh:")
    print("Batch:", latest["batch_id"])
    print("Source records:", latest["source_count"])
    print("Valid records:", latest["valid_count"])
    print("Quarantine records:", latest["quarantine_count"])

    print("\nQuality results:")
    print("Failed batch reconciliations:", bad_reconciliation)
    print("Quarantine mismatches:", quarantine_mismatches)

    assert bad_reconciliation == 0
    assert quarantine_mismatches == 0

print("\nFINAL INCREMENTAL SILVER QUALITY CHECK PASSED")


Current Silver records: 140528
Duplicate Silver IDs: 0
Invalid Silver records: 0
Successful refresh batches: 1

Latest successful refresh:
Batch: nyc_311_refresh_2026-11-03
Source records: 140660
Valid records: 140528
Quarantine records: 132

Quality results:
Failed batch reconciliations: 0
Quarantine mismatches: 0

FINAL INCREMENTAL SILVER QUALITY CHECK PASSED
